In [ ]:
from supabase import create_client
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, confusion_matrix

SUPABASE_URL     = os.environ["SUPABASE_URL"]
SUPABASE_KEY     = os.environ["SUPABASE_KEY"]
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)


def select_supabase(table, columns="*", filters=None, batch_size=1000):
     all_rows = []
     start = 0

     while True:
         query = supabase.table(table).select(columns).range(start, start + batch_size - 1)

         if filters:
             for f in filters:
                 query = query.filter(*f)

         resp = query.execute()
         rows = resp.data or []

         if not rows:
             break

         all_rows.extend(rows)

         if len(rows) < batch_size:
             break

         start += batch_size

     return all_rows

In [ ]:
response = select_supabase("servo_prices", columns="updated_at, price", filters=[("fuel_type", "eq", "U91"), ('station_id','eq','QyeyP5dO1ET4B+K47GlG67+tWN7coqUEiJPpf6NtDY8=')], batch_size=1000)

dfp = pd.DataFrame(response)

dfp.loc[pd.to_datetime(dfp["updated_at"], utc=True).dt.tz_convert("Australia/Sydney").between(datetime(2026, 5, 1, tzinfo=timezone.utc), datetime(2026, 7, 2, tzinfo=timezone.utc)), 'price'] += 16

dfp["updated_at_melb"] = pd.to_datetime(dfp["updated_at"], utc=True).dt.tz_convert("Australia/Sydney").dt.date.astype('str')
dfp["updated_at_melb_wd"] = pd.to_datetime(dfp["updated_at"], utc=True).dt.tz_convert("Australia/Sydney").dt.weekday

In [ ]:
dfagg = pd.DataFrame(dfp.groupby(['updated_at_melb','updated_at_melb_wd'])['price'].median()).reset_index()

In [ ]:

response = select_supabase("market_data", columns="date,metric, value", filters=[("metric", "in", "(brent_crude,usd_aud)")], batch_size=1000)

dfm = pd.DataFrame(response)
dfm = dfm.pivot(index="date", columns="metric", values="value").reset_index()
dfm.ffill(inplace=True) # Fill in missing values with the last known value
dfm.head()

In [ ]:
df = dfm.merge(dfagg, left_on='date', right_on="updated_at_melb", how="left")

In [ ]:
# --- Brent crude features (differenced, not raw levels) ---
brent_y = df["brent_crude"].shift(1)  # "yesterday's" brent (avoid same-day leakage)
for lag in range(1, 8):
    df[f"brent_diff{lag}"] = brent_y - df["brent_crude"].shift(lag + 1)

df["brent_ma3"]   = brent_y.rolling(3).mean() - brent_y
df["brent_ma7"]   = brent_y.rolling(7).mean() - brent_y
df["brent_trend"] = brent_y - df["brent_crude"].shift(5)

# --- USD/AUD features (differenced) ---
usd_y = df["usd_aud"].shift(1)
df["usd_aud_diff1"] = usd_y - df["usd_aud"].shift(2)
df["usd_aud_trend"] = usd_y - df["usd_aud"].shift(5)

# --- U91 price features (differenced) ---
df["u91_diff1"]    = df["price"].shift(1) - df["price"].shift(2)
df["u91_diff2"]    = df["price"].shift(2) - df["price"].shift(3)
df["u91_momentum"] = df["price"].shift(1) - df["price"].shift(3)

# --- Melbourne fuel cycle feature: days since the last sharp price rise ---
def days_since_price_rise(price, threshold=1.0):
    diffs = price.diff()
    counter = 0
    result = []
    for d in diffs:
        if pd.notna(d) and d > threshold:
            counter = 0
        else:
            counter += 1
        result.append(counter)
    return result

df["days_since_cycle_rise"] = days_since_price_rise(df["price"].shift(1))

df["is_weekend"]   = (df["updated_at_melb_wd"] >= 5).astype(int)

# One-hot encode day of week
for d in range(7):
    df[f"dow_{d}"] = (df["updated_at_melb_wd"] == d).astype(int)

df.set_index("date", inplace=True)
drop_cols = ["updated_at_melb", "updated_at_melb_wd", "brent_crude", "usd_aud"]

df.drop(columns=drop_cols, inplace=True)

df.tail(5)

In [ ]:
df["target"] = (df["price"].shift(-1) > df["price"]).astype(int)
df = df.dropna(subset=["price", "target"])

feature_cols = [c for c in df.columns if c not in ["price", "target"]]
df = df.dropna(subset=feature_cols)
 
features = df[feature_cols]
target   = df["target"]
print(f"Features shape: {features.shape}, Target shape: {target.shape}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Chronological split (no shuffling) — this is a time series, a random split would leak
# information from neighboring days between train and test.
split_idx = int(len(df) * 0.75)
X_train, X_test = features.iloc[:split_idx], features.iloc[split_idx:]
y_train, y_test = target.iloc[:split_idx], target.iloc[split_idx:]

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

baseline_majority = y_train.mode()[0]
print("Baseline (predict majority class every time) accuracy:", (y_test == baseline_majority).mean())

In [ ]:
pd.Series(model.feature_importances_, index=features.columns).sort_values(ascending=False).plot(kind="barh", figsize=(10, 6))

In [ ]:
model.predict(pd.DataFrame(df[features.columns].iloc[-1]).T)